<div style="background:#512888; color:white; border-bottom:7px solid #C6A15B; border-radius:12px; padding:36px 33px; font-family:Arial,sans-serif;">
<div style="font-size:11px; letter-spacing:2.4px; color:#F0DDAE; font-weight:700;">STEPHEN F. AUSTIN STATE UNIVERSITY &nbsp; / &nbsp; COMPUTER SCIENCE</div>
<div style="font-size:37px; font-weight:800; margin-top:13px; line-height:1.2;">Pandas <code style="background:transparent;color:#FFFFFF;font-weight:700;">pd.pivot_table()</code></div>
<div style="font-size:18px; margin-top:10px; color:#F3EBF8;">A focused, step-by-step lecture tutorial</div>
<div style="font-size:13px; color:#F0DDAE; margin-top:27px;">CSCI 1462 · Introduction to Data Analytics · Lecture Notes</div>
</div>
<p style="font-family:Arial,sans-serif;color:#594F65;">This notebook focuses <b>exclusively on <code>pd.pivot_table()</code></b>. All examples are self-contained, and there are no assignment questions.</p>

### Learning outcomes

By the end of this lecture, you will be able to:

- Describe how `pd.pivot_table()` turns individual records into summaries.
- Choose `index`, `columns`, `values`, and `aggfunc` correctly.
- Create summaries with totals, averages, counts, and multiple statistics.
- Understand `fill_value`, `margins`, and multiple grouping levels.
- Interpret a pivot-table result and avoid common mistakes.

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">01 &nbsp;·&nbsp; THE BIG PICTURE</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">What does a pivot table do?</div></div>

A **pivot table** combines many original rows into a smaller summary table.

Imagine that a campus bookstore records one row every time a product is purchased. We might want **total revenue by region and product**, instead of reading every transaction.

| Original transaction | Region | Product | Revenue |
|:--|:--|:--|--:|
| Transaction 1 | East | Pen | 10 |
| Transaction 2 | East | Pen | 4 |
| Transaction 3 | West | Pen | 16 |

A pivot table with `aggfunc="sum"` produces **East + Pen = 14** and **West + Pen = 16**.

The operation follows three steps: **group the rows → calculate a statistic → arrange the result**.

<div style="margin:14px 0 21px;padding:14px 18px;border-radius:7px;background:#FFF9EE;border-left:4px solid #C6A15B;font-family:Arial,sans-serif;line-height:1.6;color:#302042;"><b>Core idea</b><br/>The value in a pivot-table cell may summarize <strong>multiple rows</strong> from the original dataset.</div>

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">02 &nbsp;·&nbsp; START HERE</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Build our sample DataFrame</div></div>

We will use fictional campus bookstore transactions. Each row contains a month, region, product, number of units sold, and revenue.

The data has repeated region–product combinations, which makes it useful for demonstrating aggregation. Run this cell first; later cells reuse `sales`.

In [2]:
import pandas as pd
from IPython.display import display

sales = pd.DataFrame({
    "Month":   ["Jan", "Jan", "Jan", "Jan", "Jan", "Jan",
                "Feb", "Feb", "Feb", "Feb", "Feb", "Feb"],
    "Region":  ["East", "East", "East", "West", "West", "North",
                "East", "East", "West", "West", "North", "North"],
    "Product": ["Pen", "Pen", "Notebook", "Pen", "Mug", "Pen",
                "Pen", "Mug", "Pen", "Notebook", "Notebook", "Mug"],
    "Quantity": [5, 2, 2, 8, 3, 4, 6, 2, 3, 1, 3, 1],
    "Revenue":  [10, 4, 24, 16, 24, 8, 12, 16, 6, 12, 36, 8]
})

print('Original bookstore transaction table:')
display(sales)
print('Transactions:', len(sales))

Original bookstore transaction table:


,Month,Region,Product,Quantity,Revenue
0,Jan,East,Pen,5,10
1,Jan,East,Pen,2,4
2,Jan,East,Notebook,2,24
3,Jan,West,Pen,8,16
4,Jan,West,Mug,3,24
5,Jan,North,Pen,4,8
6,Feb,East,Pen,6,12
7,Feb,East,Mug,2,16
8,Feb,West,Pen,3,6
9,Feb,West,Notebook,1,12


Transactions: 12


<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">03 &nbsp;·&nbsp; PARAMETERS</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">The syntax of pd.pivot_table()</div></div>

The function is written as **`pd.pivot_table(data, ...)`**:

```python
pd.pivot_table(
    data=sales,          # the original DataFrame
    index="Region",      # categories displayed as rows
    columns="Product",   # categories displayed as columns
    values="Revenue",    # numbers to summarize
    aggfunc="sum"        # summary calculation
)
```

| Argument | What it controls | Example |
|:--|:--|:--|
| `data` | Original DataFrame | `sales` |
| `index` | Row categories | `"Region"` |
| `columns` | Column categories | `"Product"` |
| `values` | Measurements summarized | `"Revenue"` |
| `aggfunc` | Calculation performed within each group | `"sum"` |

The `columns` argument is **optional**. You can build a summary grouped only by rows.

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">04 &nbsp;·&nbsp; ONE GROUPING COLUMN</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Start with a one-dimensional summary</div></div>

First, summarize the total bookstore revenue in each region. Since we only want rows by region, there is no need to supply `columns`.

In [3]:
revenue_by_region = pd.pivot_table(
    data=sales,
    index="Region",
    values="Revenue",
    aggfunc="sum"
)

display(revenue_by_region)

,Revenue
Region,
East,66
North,52
West,58


<div style="margin:14px 0 21px;padding:14px 18px;border-radius:7px;background:#F7F3FB;border-left:4px solid #512888;font-family:Arial,sans-serif;line-height:1.6;color:#302042;"><b>How to read it</b><br/>Each region appears once. For each region, pandas adds together the <code>Revenue</code> from every matching transaction.</div>

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">05 &nbsp;·&nbsp; TWO-DIMENSIONAL PIVOT</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Turn products into columns</div></div>

Use `index` and `columns` together to compute revenue for **each region–product combination**. Observe that several transactions can contribute to the same cell.

In [4]:
by_region_product = pd.pivot_table(
    data=sales,
    index="Region",
    columns="Product",
    values="Revenue",
    aggfunc="sum"
)

display(by_region_product)

Product,Mug,Notebook,Pen
Region,,,
East,16,24,26
North,8,36,8
West,24,12,22


For example, the **East / Pen** cell contains `10 + 4 + 12 = 26`. The **West / Pen** cell contains `16 + 6 = 22`.

A `NaN` entry means that the dataset has no records for that particular region–product combination.

In [5]:
print('Original records contributing to East / Pen:')
display(sales.loc[(sales['Region'] == 'East') & (sales['Product'] == 'Pen')])

print('Corresponding pivot-table value:', by_region_product.loc['East', 'Pen'])

Original records contributing to East / Pen:


,Month,Region,Product,Quantity,Revenue
0,Jan,East,Pen,5,10
1,Jan,East,Pen,2,4
6,Feb,East,Pen,6,12


Corresponding pivot-table value: 26


<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">06 &nbsp;·&nbsp; SUM · MEAN · COUNT</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Change the aggregation function</div></div>

The `aggfunc` argument determines how values from the same group are combined.

| Aggregation | Meaning |
|:--|:--|
| `"sum"` | Total value |
| `"mean"` | Arithmetic average |
| `"count"` | Number of nonmissing values |
| `"min"` | Smallest value |
| `"max"` | Largest value |

**If `aggfunc` is omitted, pandas uses `"mean"` by default.**

In [6]:
print('A. Total revenue by product')
display(pd.pivot_table(sales, index='Product', values='Revenue', aggfunc='sum'))

print('B. Average revenue per transaction by product')
display(pd.pivot_table(sales, index='Product', values='Revenue', aggfunc='mean'))

print('C. Number of recorded revenue values by product')
display(pd.pivot_table(sales, index='Product', values='Revenue', aggfunc='count'))

A. Total revenue by product


,Revenue
Product,
Mug,48
Notebook,72
Pen,56


B. Average revenue per transaction by product


,Revenue
Product,
Mug,16.000000
Notebook,24.000000
Pen,9.333333


C. Number of recorded revenue values by product


,Revenue
Product,
Mug,3
Notebook,3
Pen,6


<div style="margin:14px 0 21px;padding:14px 18px;border-radius:7px;background:#FFF9EE;border-left:4px solid #C6A15B;font-family:Arial,sans-serif;line-height:1.6;color:#302042;"><b>Count is not quantity sold</b><br/><code>aggfunc="count"</code> counts the number of nonmissing values in the selected measurement column. It does <strong>not</strong> add the <code>Quantity</code> column.</div>

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">7 &nbsp;·&nbsp; MULTIPLE VALUES</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Summarize several measurements</div></div>

The `values` argument can also receive a list. Here, each region is summarized using **units sold** and **revenue**.

In [7]:
two_measurements = pd.pivot_table(
    sales,
    index='Region',
    values=['Quantity', 'Revenue'],
    aggfunc='sum'
)

display(two_measurements)

,Quantity,Revenue
Region,,
East,17,66
North,8,52
West,15,58


Both measurements use the same aggregation (`sum`) in this example.

<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">8 &nbsp;·&nbsp; AGGFUNC DICTIONARY</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Different calculations for different columns</div></div>

Supply a dictionary to `aggfunc` when you want different calculations for different measurements.

For example, calculate **total units** sold but **average revenue per transaction** in each region.

In [8]:
different_calculations = pd.pivot_table(
    sales,
    index='Region',
    values=['Quantity', 'Revenue'],
    aggfunc={'Quantity': 'sum', 'Revenue': 'mean'}
)

display(different_calculations)

,Quantity,Revenue
Region,,
East,17,13.200000
North,8,17.333333
West,15,14.500000


<div style="margin:28px 0 13px; background:#512888; color:white; border-radius:9px; border-left:7px solid #C6A15B; padding:15px 21px; font-family:Arial,sans-serif;">
    <div style="font-size:11px;letter-spacing:2px;color:#F0DDAE;font-weight:700;">9 &nbsp;·&nbsp; LIST OF AGGREGATIONS</div>
    <div style="font-size:23px;font-weight:750;margin-top:4px;line-height:1.3;">Calculate multiple statistics at once</div></div>

A list in `aggfunc` creates a summary with several statistics side by side. The result includes an extra level of column headings indicating which statistic is shown.

In [9]:
summary_statistics = pd.pivot_table(
    sales,
    index='Region',
    values='Revenue',
    aggfunc=['sum', 'mean', 'count', 'min', 'max']
)

display(summary_statistics)

,sum,mean,count,min,max
,Revenue,Revenue,Revenue,Revenue,Revenue
Region,,,,,
East,66,13.200000,5,4,24
North,52,17.333333,3,8,36
West,58,14.500000,4,6,24


<div style="margin:24px 0 0; border-top:5px solid #C6A15B; background:#F5F1FA; padding:20px 22px; border-radius:8px; font-family:Arial,sans-serif; color:#302042;">
<span style="font-size:19px; font-weight:750;">Key takeaway</span><br/><br/>
<code>pd.pivot_table()</code> summarizes records by categories. You decide which categories form the <strong>rows and columns</strong>, which measurements to summarize, and <strong>which aggregation</strong> to apply.
</div>

**Reference:** [Official pandas documentation — `pandas.pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).